In [0]:
%sql
CREATE OR REPLACE TEMP VIEW base_mb52_4128 AS
SELECT *
FROM prd_procurement.corp_curated.vw_ds_log_mb52_estoque_por_deposito
WHERE TRIM(CAST(cod_centro AS STRING)) = '4128';

In [0]:
%sql
SELECT
    COUNT(*) AS linhas_total,
    COUNT(DISTINCT cod_material) AS materiais_distintos,
    COUNT(DISTINCT cod_deposito) AS depositos_distintos,
    COUNT(DISTINCT dateingest) AS datas_ingestao
FROM base_mb52_4128;

In [0]:
%sql
DESCRIBE EXTENDED
prd_procurement.corp_curated.vw_ds_log_mb52_estoque_por_deposito;

In [0]:
%sql
SELECT
    CAST(dateingest AS DATE) AS data_ingestao,
    COUNT(*) AS linhas,
    COUNT(DISTINCT cod_material) AS materiais,
    COUNT(DISTINCT cod_deposito) AS depositos,
    COUNT(
        DISTINCT CONCAT_WS(
            '|',
            REGEXP_REPLACE(
                TRIM(CAST(cod_material AS STRING)),
                '^0+',
                ''
            ),
            UPPER(TRIM(CAST(cod_deposito AS STRING)))
        )
    ) AS chaves_material_deposito
FROM base_mb52_4128
GROUP BY CAST(dateingest AS DATE)
ORDER BY data_ingestao;

In [0]:
%sql
SELECT
    CAST(dateingest AS DATE) AS data_ingestao,

    COUNT(*) AS linhas_total,

    COUNT_IF(
        COALESCE(qt_utilizacao_livre, 0) = 0
        AND COALESCE(vl_utilizacao_livre, 0) = 0
    ) AS linhas_qtd_valor_zero,

    COUNT_IF(
        COALESCE(qt_utilizacao_livre, 0) <> 0
        OR COALESCE(vl_utilizacao_livre, 0) <> 0
    ) AS linhas_qtd_ou_valor_nao_zero,

    ROUND(
        100.0 * COUNT_IF(
            COALESCE(qt_utilizacao_livre, 0) = 0
            AND COALESCE(vl_utilizacao_livre, 0) = 0
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS pct_linhas_zeradas

FROM base_mb52_4128
GROUP BY CAST(dateingest AS DATE)
ORDER BY data_ingestao;

In [0]:
%sql
SELECT
    CASE
        WHEN COALESCE(qt_utilizacao_livre, 0) = 0
         AND COALESCE(vl_utilizacao_livre, 0) = 0
            THEN 'QTD_ZERO_VALOR_ZERO'

        WHEN COALESCE(qt_utilizacao_livre, 0) <> 0
         AND COALESCE(vl_utilizacao_livre, 0) = 0
            THEN 'QTD_NAO_ZERO_VALOR_ZERO'

        WHEN COALESCE(qt_utilizacao_livre, 0) = 0
         AND COALESCE(vl_utilizacao_livre, 0) <> 0
            THEN 'QTD_ZERO_VALOR_NAO_ZERO'

        ELSE 'QTD_NAO_ZERO_VALOR_NAO_ZERO'
    END AS classificacao,

    COUNT(*) AS linhas,

    COUNT(
        DISTINCT CONCAT_WS(
            '|',
            REGEXP_REPLACE(
                TRIM(CAST(cod_material AS STRING)),
                '^0+',
                ''
            ),
            UPPER(TRIM(CAST(cod_deposito AS STRING)))
        )
    ) AS chaves

FROM base_mb52_4128
GROUP BY 1
ORDER BY linhas DESC;

In [0]:
%sql
WITH chaves AS (
    SELECT
        CAST(dateingest AS DATE) AS data_ingestao,

        CONCAT_WS(
            '|',
            REGEXP_REPLACE(
                TRIM(CAST(cod_material AS STRING)),
                '^0+',
                ''
            ),
            UPPER(TRIM(CAST(cod_deposito AS STRING)))
        ) AS chave,

        COUNT(*) AS quantidade_linhas

    FROM base_mb52_4128

    GROUP BY
        CAST(dateingest AS DATE),
        CONCAT_WS(
            '|',
            REGEXP_REPLACE(
                TRIM(CAST(cod_material AS STRING)),
                '^0+',
                ''
            ),
            UPPER(TRIM(CAST(cod_deposito AS STRING)))
        )
)

SELECT
    data_ingestao,
    COUNT(*) AS chaves_total,
    COUNT_IF(quantidade_linhas = 1) AS chaves_unicas,
    COUNT_IF(quantidade_linhas > 1) AS chaves_repetidas,
    SUM(
        CASE
            WHEN quantidade_linhas > 1
            THEN quantidade_linhas - 1
            ELSE 0
        END
    ) AS linhas_excedentes,
    MAX(quantidade_linhas) AS maior_repeticao

FROM chaves
GROUP BY data_ingestao
ORDER BY data_ingestao;

In [0]:
%sql
SELECT
    CAST(dateingest AS DATE) AS data_ingestao,
    REGEXP_REPLACE(
        TRIM(CAST(cod_material AS STRING)),
        '^0+',
        ''
    ) AS material_normalizado,
    UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,
    COUNT(*) AS quantidade_linhas

FROM base_mb52_4128

GROUP BY
    CAST(dateingest AS DATE),
    REGEXP_REPLACE(
        TRIM(CAST(cod_material AS STRING)),
        '^0+',
        ''
    ),
    UPPER(TRIM(CAST(cod_deposito AS STRING)))

HAVING COUNT(*) > 1

ORDER BY
    quantidade_linhas DESC,
    material_normalizado,
    deposito;

In [0]:
%sql
WITH chaves AS (
    SELECT
        REGEXP_REPLACE(
            TRIM(CAST(cod_material AS STRING)),
            '^0+',
            ''
        ) AS material_normalizado,

        UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,

        CAST(dateingest AS DATE) AS data_ingestao

    FROM base_mb52_4128
)

SELECT
    material_normalizado,
    deposito,
    COUNT(*) AS linhas,
    COUNT(DISTINCT data_ingestao) AS snapshots,
    MIN(data_ingestao) AS primeira_data,
    MAX(data_ingestao) AS ultima_data,
    CONCAT_WS(
        ', ',
        SORT_ARRAY(
            COLLECT_SET(CAST(data_ingestao AS STRING))
        )
    ) AS datas_encontradas

FROM chaves

GROUP BY
    material_normalizado,
    deposito

HAVING COUNT(DISTINCT data_ingestao) > 1

ORDER BY
    snapshots DESC,
    linhas DESC,
    material_normalizado,
    deposito;

In [0]:
%sql
WITH base_normalizada AS (
    SELECT
        REGEXP_REPLACE(
            TRIM(CAST(cod_material AS STRING)),
            '^0+',
            ''
        ) AS material_normalizado,

        UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,

        CAST(dateingest AS DATE) AS data_ingestao,

        qt_utilizacao_livre,
        vl_utilizacao_livre,
        sg_unidade_medida_basica,
        tp_material,
        tp_grupo_mercadorias,
        desc_material

    FROM base_mb52_4128
),

comparacao AS (
    SELECT
        material_normalizado,
        deposito,

        COUNT(*) AS linhas,
        COUNT(DISTINCT data_ingestao) AS snapshots,

        COUNT(
            DISTINCT CONCAT_WS(
                '|',
                COALESCE(CAST(qt_utilizacao_livre AS STRING), 'NULL'),
                COALESCE(CAST(vl_utilizacao_livre AS STRING), 'NULL'),
                COALESCE(TRIM(sg_unidade_medida_basica), 'NULL'),
                COALESCE(TRIM(tp_material), 'NULL'),
                COALESCE(TRIM(tp_grupo_mercadorias), 'NULL'),
                COALESCE(TRIM(desc_material), 'NULL')
            )
        ) AS versoes_negocio

    FROM base_normalizada

    GROUP BY
        material_normalizado,
        deposito
)

SELECT
    material_normalizado,
    deposito,
    linhas,
    snapshots,
    versoes_negocio,

    CASE
        WHEN snapshots > 1
         AND versoes_negocio = 1
            THEN 'REPETICAO_IDENTICA_ENTRE_CARGAS'

        WHEN snapshots > 1
         AND versoes_negocio > 1
            THEN 'ALTERACAO_HISTORICA_REAL'

        WHEN linhas > 1
            THEN 'REPETICAO_DENTRO_DO_SNAPSHOT'

        ELSE 'CHAVE_UNICA'
    END AS classificacao

FROM comparacao

WHERE linhas > 1

ORDER BY
    classificacao,
    linhas DESC,
    material_normalizado,
    deposito;

In [0]:
%sql
WITH datas AS (
    SELECT
        MIN(CAST(dateingest AS DATE)) AS primeira_data,
        MAX(CAST(dateingest AS DATE)) AS ultima_data
    FROM base_mb52_4128
),

chaves AS (
    SELECT DISTINCT
        CAST(dateingest AS DATE) AS data_ingestao,

        CONCAT_WS(
            '|',
            REGEXP_REPLACE(
                TRIM(CAST(cod_material AS STRING)),
                '^0+',
                ''
            ),
            UPPER(TRIM(CAST(cod_deposito AS STRING)))
        ) AS chave

    FROM base_mb52_4128
),

primeira_carga AS (
    SELECT chave
    FROM chaves
    CROSS JOIN datas
    WHERE data_ingestao = primeira_data
),

ultima_carga AS (
    SELECT chave
    FROM chaves
    CROSS JOIN datas
    WHERE data_ingestao = ultima_data
)

SELECT
    CASE
        WHEN p.chave IS NOT NULL
         AND u.chave IS NOT NULL
            THEN 'EXISTE_NAS_DUAS_CARGAS'

        WHEN p.chave IS NOT NULL
         AND u.chave IS NULL
            THEN 'EXISTE_SOMENTE_NA_PRIMEIRA'

        WHEN p.chave IS NULL
         AND u.chave IS NOT NULL
            THEN 'EXISTE_SOMENTE_NA_ULTIMA'
    END AS situacao,

    COUNT(*) AS chaves

FROM primeira_carga p
FULL OUTER JOIN ultima_carga u
    ON p.chave = u.chave

GROUP BY 1
ORDER BY 1;

In [0]:
%sql
WITH datas AS (
    SELECT
        MIN(CAST(dateingest AS DATE)) AS primeira_data,
        MAX(CAST(dateingest AS DATE)) AS ultima_data
    FROM base_mb52_4128
),

primeira_carga AS (
    SELECT
        REGEXP_REPLACE(
            TRIM(CAST(cod_material AS STRING)),
            '^0+',
            ''
        ) AS material,

        UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,

        qt_utilizacao_livre,
        vl_utilizacao_livre,

        CAST(dateingest AS DATE) AS data_ingestao

    FROM base_mb52_4128
    CROSS JOIN datas

    WHERE CAST(dateingest AS DATE) = primeira_data
),

ultima_carga AS (
    SELECT DISTINCT
        REGEXP_REPLACE(
            TRIM(CAST(cod_material AS STRING)),
            '^0+',
            ''
        ) AS material,

        UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito

    FROM base_mb52_4128
    CROSS JOIN datas

    WHERE CAST(dateingest AS DATE) = ultima_data
)

SELECT
    p.material,
    p.deposito,
    p.qt_utilizacao_livre,
    p.vl_utilizacao_livre,
    p.data_ingestao

FROM primeira_carga p

LEFT ANTI JOIN ultima_carga u
    ON p.material = u.material
   AND p.deposito = u.deposito

ORDER BY
    ABS(COALESCE(p.vl_utilizacao_livre, 0)) DESC,
    ABS(COALESCE(p.qt_utilizacao_livre, 0)) DESC;

In [0]:
%sql
SELECT
    CAST(dateingest AS DATE) AS data_ingestao,
    UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,

    COUNT(*) AS linhas,

    COUNT(
        DISTINCT REGEXP_REPLACE(
            TRIM(CAST(cod_material AS STRING)),
            '^0+',
            ''
        )
    ) AS materiais,

    COUNT_IF(
        COALESCE(qt_utilizacao_livre, 0) = 0
        AND COALESCE(vl_utilizacao_livre, 0) = 0
    ) AS linhas_zeradas,

    COUNT_IF(
        COALESCE(qt_utilizacao_livre, 0) <> 0
        OR COALESCE(vl_utilizacao_livre, 0) <> 0
    ) AS linhas_nao_zeradas

FROM base_mb52_4128

GROUP BY
    CAST(dateingest AS DATE),
    UPPER(TRIM(CAST(cod_deposito AS STRING)))

ORDER BY
    data_ingestao,
    materiais DESC;

In [0]:
%sql
SELECT
    UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,
    COUNT(DISTINCT CAST(dateingest AS DATE)) AS snapshots,
    MIN(CAST(dateingest AS DATE)) AS primeira_data,
    MAX(CAST(dateingest AS DATE)) AS ultima_data,
    COUNT(*) AS linhas,
    COUNT(DISTINCT cod_material) AS materiais

FROM base_mb52_4128

GROUP BY
    UPPER(TRIM(CAST(cod_deposito AS STRING)))

HAVING COUNT(DISTINCT CAST(dateingest AS DATE)) = 1

ORDER BY
    materiais DESC,
    deposito;

In [0]:
%sql
WITH preenchimento AS (
    SELECT
        COUNT(*) AS total,

        COUNT_IF(cod_material IS NULL OR TRIM(CAST(cod_material AS STRING)) = '')
            AS nulo_cod_material,

        COUNT_IF(cod_deposito IS NULL OR TRIM(CAST(cod_deposito AS STRING)) = '')
            AS nulo_cod_deposito,

        COUNT_IF(qt_utilizacao_livre IS NULL)
            AS nulo_qt_utilizacao_livre,

        COUNT_IF(vl_utilizacao_livre IS NULL)
            AS nulo_vl_utilizacao_livre,

        COUNT_IF(
            sg_unidade_medida_basica IS NULL
            OR TRIM(CAST(sg_unidade_medida_basica AS STRING)) = ''
        ) AS nulo_unidade,

        COUNT_IF(
            tp_material IS NULL
            OR TRIM(CAST(tp_material AS STRING)) = ''
        ) AS nulo_tipo_material,

        COUNT_IF(
            tp_grupo_mercadorias IS NULL
            OR TRIM(CAST(tp_grupo_mercadorias AS STRING)) = ''
        ) AS nulo_grupo_mercadorias,

        COUNT_IF(
            desc_material IS NULL
            OR TRIM(CAST(desc_material AS STRING)) = ''
        ) AS nulo_descricao,

        COUNT_IF(
            ind_eliminacao_deposito IS NULL
            OR TRIM(CAST(ind_eliminacao_deposito AS STRING)) = ''
        ) AS nulo_ind_eliminacao,

        COUNT_IF(
            tp_estoque_especial IS NULL
            OR TRIM(CAST(tp_estoque_especial AS STRING)) = ''
        ) AS nulo_tipo_estoque_especial,

        COUNT_IF(
            cod_estoque_especial IS NULL
            OR TRIM(CAST(cod_estoque_especial AS STRING)) = ''
        ) AS nulo_codigo_estoque_especial

    FROM base_mb52_4128
)

SELECT
    campo,
    nulos,
    total,
    ROUND(100.0 * nulos / NULLIF(total, 0), 2) AS pct_nulo

FROM preenchimento

LATERAL VIEW STACK(
    11,

    'cod_material',
    nulo_cod_material,

    'cod_deposito',
    nulo_cod_deposito,

    'qt_utilizacao_livre',
    nulo_qt_utilizacao_livre,

    'vl_utilizacao_livre',
    nulo_vl_utilizacao_livre,

    'sg_unidade_medida_basica',
    nulo_unidade,

    'tp_material',
    nulo_tipo_material,

    'tp_grupo_mercadorias',
    nulo_grupo_mercadorias,

    'desc_material',
    nulo_descricao,

    'ind_eliminacao_deposito',
    nulo_ind_eliminacao,

    'tp_estoque_especial',
    nulo_tipo_estoque_especial,

    'cod_estoque_especial',
    nulo_codigo_estoque_especial

) AS campo, nulos

ORDER BY pct_nulo DESC;

In [0]:
%sql
WITH datas AS (
    SELECT
        MIN(CAST(dateingest AS DATE)) AS primeira_data,
        MAX(CAST(dateingest AS DATE)) AS ultima_data,
        COUNT(DISTINCT CAST(dateingest AS DATE)) AS snapshots
    FROM base_mb52_4128
),

metricas AS (
    SELECT
        COUNT(*) AS linhas,

        COUNT(DISTINCT cod_material) AS materiais,

        COUNT(DISTINCT cod_deposito) AS depositos,

        COUNT(
            DISTINCT CONCAT_WS(
                '|',
                REGEXP_REPLACE(
                    TRIM(CAST(cod_material AS STRING)),
                    '^0+',
                    ''
                ),
                UPPER(TRIM(CAST(cod_deposito AS STRING)))
            )
        ) AS chaves,

        COUNT_IF(
            COALESCE(qt_utilizacao_livre, 0) = 0
            AND COALESCE(vl_utilizacao_livre, 0) = 0
        ) AS linhas_zeradas,

        COUNT_IF(
            COALESCE(qt_utilizacao_livre, 0) <> 0
            OR COALESCE(vl_utilizacao_livre, 0) <> 0
        ) AS linhas_nao_zeradas

    FROM base_mb52_4128
)

SELECT
    '4128' AS centro,
    d.primeira_data,
    d.ultima_data,
    d.snapshots,
    m.linhas,
    m.materiais,
    m.depositos,
    m.chaves,
    m.linhas_zeradas,
    m.linhas_nao_zeradas,

    ROUND(
        100.0 * m.linhas_zeradas
        / NULLIF(m.linhas, 0),
        2
    ) AS pct_linhas_zeradas,

    CASE
        WHEN d.snapshots = 1
            THEN 'APENAS_UM_SNAPSHOT'

        WHEN d.snapshots > 1
            THEN 'MULTIPLAS_CARGAS_NA_VIEW'

        ELSE 'SEM_DATA_DE_INGESTAO'
    END AS classificacao_temporal

FROM datas d
CROSS JOIN metricas m;